# Project Summary — XTB S.A. Risk Analysis
## Enterprise Risk Management

**Summary author:** Michal Marchwiak  
**Analysis period:** 2018–2025  
**Primary risk variable:** log-returns of **XTB.WA** shares (WSE)

---

This notebook consolidates conclusions from four project materials and presents them
in a unified, readable layout with refreshed charts and interpretations:

| # | Source file | Scope |
|---|-------------|--------|
| 1 | `prez3.ipynb` | Classical risk measures (volatility, quantiles, distribution functions), FX+CFD portfolios, EVT (GEV/GPD) |
| 2 | `prez4.ipynb` | VaR/EVaR with seven methods, backtesting (Kupiec, Christoffersen, Berkowitz, Basel) |
| 3 | `prez5.ipynb` | WSE equity portfolio optimization (Markowitz model, single-factor model) |
| 4 | `lstm_fhs_xtb.py` | Hybrid LSTM + Filtered Historical Simulation model |

### Table of contents
1. **Configuration and data**
2. **XTB.WA characteristics** — price series, log-returns, descriptive statistics
3. **Distribution fitting** — empirical vs Normal vs t-Student vs alternatives
4. **VaR and EVaR method comparison** — 7 methods, two confidence levels
5. **Backtesting and Basel Traffic Light**
6. **Extreme Value Theory (EVT)** — GEV and GPD
7. **Hybrid LSTM + FHS model**
8. **Markowitz portfolio optimization**
9. **Synthesis of conclusions and recommendations for XTB**


---
## 1. Configuration and data

We use a unified XTB color palette (signature red `#E40521`, orange accent,
dark graphite) and shared matplotlib settings so that charts throughout the notebook are consistent.


In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import yfinance as yf
import matplotlib.pyplot as plt
import matplotlib as mpl
from matplotlib.ticker import PercentFormatter
from scipy import stats
from scipy.stats import (
    norm, t as t_dist, genextreme, genpareto, kstest, chi2,
)
from scipy.optimize import minimize
from IPython.display import display, Markdown

# ---------- XTB palette (DARK MODE) ----------
# Background: dark navy/graphite; accents: signature XTB red + orange/blue.
XTB_FIG_BG   = '#0B1220'   # entire figure background (darkest)
XTB_AX_BG    = '#111827'   # axes background (slightly lighter)
XTB_INK      = '#E5E7EB'   # light text / contrast lines
XTB_GRID     = '#334155'   # grid (slate-700)

XTB_RED      = '#FF3A4D'   # brightened XTB red (better visibility on dark background)
XTB_ORANGE   = '#FBBF24'   # warmer orange accent
XTB_GRAPHITE = XTB_INK     # alias for consistency with rest of notebook — light gray
XTB_STEEL    = '#7DA8C9'   # lightened steel blue
XTB_GREEN    = '#34D399'   # green (acceptance, profit)
XTB_NAVY     = '#60A5FA'   # lightened navy -> neon blue
XTB_PALETTE  = [XTB_RED, XTB_STEEL, XTB_ORANGE, XTB_GREEN, XTB_NAVY, '#A78BFA',
                '#22D3EE', '#F472B6', '#FACC15', '#94A3B8', '#84CC16']

# ---------- Global chart styling (DARK MODE) ----------
mpl.rcParams.update({
    'figure.figsize':     (14, 5.5),
    'figure.dpi':         110,
    'savefig.dpi':        140,
    'font.size':          11,
    'font.family':        'DejaVu Sans',
    # — background colors —
    'figure.facecolor':   XTB_FIG_BG,
    'axes.facecolor':     XTB_AX_BG,
    'savefig.facecolor':  XTB_FIG_BG,
    'savefig.edgecolor':  XTB_FIG_BG,
    # — text and axis colors —
    'text.color':         XTB_INK,
    'axes.edgecolor':     XTB_INK,
    'axes.labelcolor':    XTB_INK,
    'axes.titlecolor':    XTB_INK,
    'xtick.color':        XTB_INK,
    'ytick.color':        XTB_INK,
    # — typography —
    'axes.titlesize':     13,
    'axes.titleweight':   'bold',
    'axes.labelsize':     11,
    'axes.labelweight':   'normal',
    'xtick.labelsize':    9.5,
    'ytick.labelsize':    9.5,
    # — spines —
    'axes.spines.top':    False,
    'axes.spines.right':  False,
    # — grid —
    'axes.grid':          True,
    'grid.color':         XTB_GRID,
    'grid.alpha':         0.45,
    'grid.linestyle':     '--',
    'grid.linewidth':     0.7,
    # — legend —
    'legend.frameon':     True,
    'legend.facecolor':   XTB_AX_BG,
    'legend.edgecolor':   XTB_GRID,
    'legend.labelcolor':  XTB_INK,
    'legend.framealpha':  0.92,
    'legend.fontsize':    9.5,
    # — color cycle —
    'axes.prop_cycle':    plt.cycler(color=XTB_PALETTE),
    # — patches (bars, rectangles) on dark background —
    'patch.edgecolor':    XTB_FIG_BG,
    'patch.force_edgecolor': True,
    # — boxplot / scatter edges —
    'boxplot.boxprops.color':     XTB_INK,
    'boxplot.capprops.color':     XTB_INK,
    'boxplot.whiskerprops.color': XTB_INK,
    'boxplot.flierprops.markeredgecolor': XTB_INK,
    'boxplot.medianprops.color':  XTB_RED,
})

np.random.seed(42)
print('DARK MODE configuration loaded.')
print(f'Figure background: {XTB_FIG_BG} | axes background: {XTB_AX_BG} | text: {XTB_INK}')
print(f'XTB palette: {XTB_PALETTE[:5]}')


In [ ]:
# ---------- Download XTB.WA data (2018-2025) ----------
prices_xtb = (yf.download('XTB.WA', start='2018-01-01', end='2025-12-31',
                          progress=False, auto_adjust=False)['Close']
                .dropna())
if isinstance(prices_xtb, pd.DataFrame):
    prices_xtb = prices_xtb.squeeze()
prices_xtb.name = 'XTB.WA'
log_ret = np.log(prices_xtb / prices_xtb.shift(1)).dropna()
log_ret.name = 'XTB.WA'

# Basic statistics
N = len(log_ret)
mu_n, sig_n = log_ret.mean(), log_ret.std()
df_t, loc_t, scale_t = t_dist.fit(log_ret.values)

stats_dict = {
    'Period':              f'{prices_xtb.index[0].date()} -> {prices_xtb.index[-1].date()}',
    'Number of observations':  N,
    'Mean log-return': f'{mu_n:+.5f}',
    'Std. dev. (daily)':    f'{sig_n:.5f}',
    'Std. dev. (annual)':  f'{sig_n*np.sqrt(252):.3f}',
    'Min / Max':          f'{log_ret.min():.4f} / {log_ret.max():.4f}',
    'Skewness':           f'{stats.skew(log_ret):.3f}',
    'Kurtosis (excess)':   f'{stats.kurtosis(log_ret):.3f}',
    'Fitted ν (t)':  f'{df_t:.2f}',
}
print('=' * 60)
print('XTB.WA descriptive statistics (daily log-returns)')
print('=' * 60)
for k, v in stats_dict.items():
    print(f'  {k:<20s} {v}')


---
## 2. XTB.WA characteristics

Starting point for all projects: we present the price series, log-returns,
loss histogram, and Q-Q plot against the normal distribution in one place.

> **Visualization goal:** immediately show two key properties of the series:
> *volatility clustering* and *fat tails*
> — these drive modeling decisions in subsequent sections.


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 9))
fig.suptitle('XTB.WA — return series characteristics (2018–2025)',
             fontsize=15, fontweight='bold', y=1.00)

# --- (a) Closing price ---
ax = axes[0, 0]
ax.plot(prices_xtb.index, prices_xtb.values, color=XTB_NAVY, lw=1.1)
ax.fill_between(prices_xtb.index, prices_xtb.values, alpha=0.08, color=XTB_NAVY)
ax.set_title('XTB.WA closing price (PLN)')
ax.set_ylabel('Price [PLN]')

# --- (b) Log-returns ---
ax = axes[0, 1]
ax.plot(log_ret.index, log_ret.values, color=XTB_RED, lw=0.55, alpha=0.85)
ax.axhline(0, color=XTB_GRAPHITE, lw=0.6)
# Mark largest extremes
top_neg = log_ret.nsmallest(3)
for d, v in top_neg.items():
    ax.annotate(f'{v:+.1%}', xy=(d, v), xytext=(8, -10), textcoords='offset points',
                fontsize=8.5, color=XTB_RED, fontweight='bold')
ax.set_title('Daily log-returns (volatility clustering)')
ax.set_ylabel('log-return')
ax.yaxis.set_major_formatter(PercentFormatter(xmax=1.0, decimals=0))

# --- (c) Return histogram vs Normal vs t-Student ---
ax = axes[1, 0]
bins = np.linspace(log_ret.min(), log_ret.max(), 100)
ax.hist(log_ret, bins=bins, density=True, alpha=0.55,
        color=XTB_STEEL, edgecolor='white', linewidth=0.4, label='Empirical')
x_grid = np.linspace(log_ret.min(), log_ret.max(), 600)
ax.plot(x_grid, norm.pdf(x_grid, mu_n, sig_n),
        color=XTB_GRAPHITE, lw=1.8, ls='--', label='Normal')
ax.plot(x_grid, t_dist.pdf(x_grid, df_t, loc_t, scale_t),
        color=XTB_RED, lw=2, label=f't-Student (nu={df_t:.2f})')
ax.set_xlim(-0.12, 0.12)
ax.set_title('Log-return histogram vs fitted distributions')
ax.set_xlabel('log-return')
ax.set_ylabel('density')
ax.xaxis.set_major_formatter(PercentFormatter(xmax=1.0, decimals=0))
ax.legend(loc='upper left')

# --- (d) Q-Q plot vs Normal ---
ax = axes[1, 1]
stats.probplot(log_ret.values, dist='norm', plot=ax)
ax.get_lines()[0].set_marker('o')
ax.get_lines()[0].set_markersize(3)
ax.get_lines()[0].set_color(XTB_RED)
ax.get_lines()[0].set_alpha(0.55)
ax.get_lines()[1].set_color(XTB_GRAPHITE)
ax.get_lines()[1].set_lw(1.2)
ax.set_title('Q-Q plot vs normal distribution')
ax.set_xlabel('theoretical quantiles')
ax.set_ylabel('empirical quantiles')

plt.tight_layout()
plt.show()


### Conclusions — characteristics chart

1. **XTB.WA price rose roughly 10-fold** over 2018–2025 (from ~5 PLN to over 80 PLN).
   The company went from a small broker to a WIG20 constituent — this implies a change in the risk regime
   and justifies models that adapt to **time-varying volatility** (EWMA, GARCH, LSTM).

2. **Volatility clustering** — clearly visible bands of high amplitude (COVID crash 2020,
   extreme moves in 2022–2024 linked to XTB revenue fluctuations).
   The static **Plain HS** method ignores this dynamics and clusters breaches (confirmed
   by the Christoffersen test in Section 5).

3. **Fat tails** — the histogram lies significantly above the normal curve in the tails.
   - **Excess kurtosis ≈ 25** (vs 0 for the normal distribution) — an extreme value.
   - The fitted t-Student distribution has **ν ≈ 2.56**, meaning variance is finite,
     but **the third moment and higher do not exist** — a formal indicator of a very heavy tail.

4. **Q-Q plot** shows typical S-shaped deviations from normality: extreme negative returns
   are **many times more likely** than the normal distribution would suggest.
   This is a key argument for rejecting the Param Normal method when estimating VaR 99%.

5. **Negative skew (-0.44)** — negative extremes are larger than positive ones.
   Implication: symmetric risk measures (parametric t with `loc=0`) systematically
   underestimate the left tail — hence the motivation for **FHS** (using the empirical quantile of residuals).


---
## 3. Distribution fitting — Kolmogorov-Smirnov test

From prez3.ipynb we know that the K-S test **rejects the normal distribution** for XTB.WA (p < 10^-4),
as well as t-Student (p ≈ 0.003). Among alternatives, **Johnson SU** achieved the best fit,
but it too was rejected on the full sample.

Below we compile p-values for key distributions for XTB.WA, so we can see in one place
**why no parametric distribution fits fully** — which in turn justifies
*semi-parametric* approaches (FHS, EVT, LSTM-FHS).


In [ ]:
from scipy.stats import laplace, skewnorm, gennorm, johnsonsu, nct, levy_stable

r = log_ret.values
candidates = [
    ('Normal',           norm,       norm.fit(r)),
    ('t-Student',        t_dist,     t_dist.fit(r)),
    ('Laplace',          laplace,    laplace.fit(r)),
    ('Skew-Normal',      skewnorm,   skewnorm.fit(r)),
    ('Gen. Normal',      gennorm,    gennorm.fit(r)),
    ('Johnson SU',       johnsonsu,  johnsonsu.fit(r)),
    ('NoncentralT',      nct,        nct.fit(r)),
]

rows = []
for name, dist_obj, params in candidates:
    ks_stat, ks_p = kstest(r, dist_obj.cdf, args=params)
    rows.append({'Distribution': name,
                 'KS stat': ks_stat,
                 'p-value': ks_p,
                 'Verdict': 'ACCEPT' if ks_p >= 0.05 else 'REJECT'})
df_ks = pd.DataFrame(rows).sort_values('p-value', ascending=False).reset_index(drop=True)

display(df_ks.style.format({'KS stat': '{:.4f}', 'p-value': '{:.4f}'}))

# p-value chart
fig, ax = plt.subplots(figsize=(13, 4.5))
colors = [XTB_GREEN if w == 'ACCEPT' else XTB_RED for w in df_ks['Verdict']]
bars = ax.barh(df_ks['Distribution'], df_ks['p-value'], color=colors,
               edgecolor='white', height=0.62)
ax.axvline(0.05, color=XTB_GRAPHITE, lw=1.4, ls='--', label='Significance level α = 0.05')
for bar, p in zip(bars, df_ks['p-value']):
    ax.text(bar.get_width() + 0.005, bar.get_y() + bar.get_height()/2,
            f'p = {p:.4f}', va='center', fontsize=9.5)
ax.set_xlim(0, max(df_ks['p-value'].max() * 1.18, 0.08))
ax.set_xlabel('K-S test p-value')
ax.set_title('Kolmogorov-Smirnov test: which distributions describe XTB.WA log-returns?')
ax.invert_yaxis()
ax.legend(loc='lower right')
plt.tight_layout()
plt.show()


### Conclusions — distribution fitting

1. **All parametric distributions are rejected** at α = 0.05 for XTB.WA.
   Relative ordering: Johnson SU ≈ NoncentralT > t-Student >> Normal ≈ Laplace.

2. **The scale of rejection determines model risk:**
   - the **normal** distribution has p-value on the order of 10^-12 — using it for VaR 99% is practically an error;
   - **t-Student**, although formally rejected, has drastically better fit (1000×
     higher p-value than normal) and is an **acceptable choice** for parametric VaR
     **at the 5% level**, but at 1% leads to conservative estimates (see ν = 2.56,
     very heavy tail).

3. **Modeling implication:**
   - for *risk reporting* → best to use t-Student or a nonparametric distribution;
   - for *regulatory* tasks (Basel/FRTB) → a dynamic approach (EWMA/GARCH/LSTM) is required
     to capture volatility clustering, which a static probability distribution
     can never reproduce;
   - for *capital allocation* → bootstrap CIs for VaR matter (Section 4.4 of prez4)
     shows that point VaR 99% uncertainty is 10–25%.


---
## 4. VaR and EVaR method comparison (XTB.WA)

In prez4, VaR and EVaR were computed with seven methods. Below we reconstruct the table
and show an aggregate comparison chart.

**Brief method overview:**

| Method | Idea | Advantages | Disadvantages |
|---|---|---|---|
| **Param Normal** | fit N(μ, σ²) | analytical | ignores fat tails |
| **Param t-Student** | fit t_ν | handles fat tails | symmetric around μ |
| **Plain HS** | empirical quantile | no distributional assumptions | no dynamics |
| **Weighted HS (BRW)** | weights increasing over time | responds to regime | arbitrary λ choice |
| **FHS GARCH** | empirical residuals × σ_GARCH | dynamics + fat tails | requires GARCH MLE |
| **EWMA + Hill** | EWMA + Hill estimator | EVT on residuals | sensitivity to k choice |
| **MC t / GBM / ARMA-GARCH** | simulation | flexibility, multi-day | simulation noise |


In [ ]:
# Reconstruction of key numbers from prez4 (in-sample, full sample)
from arch import arch_model

# --- 1a. Parametric ---
mu_n, sig_n = r.mean(), r.std()
df_t, loc_t, scale_t = t_dist.fit(r)

# --- 1b.i Plain HS ---
# --- 1b.ii Weighted HS (BRW) ---
lam_brw = 0.995
n = len(r)
ages = np.arange(1, n + 1)
w_brw = lam_brw**(n - ages) * (1 - lam_brw) / (1 - lam_brw**n)

def wquant(values, weights, q):
    order = np.argsort(values)
    vals = values[order]; wts = weights[order]
    cum = np.cumsum(wts)
    idx = np.searchsorted(cum, q)
    return vals[min(idx, len(vals)-1)]

# --- 1b.iii FHS GARCH-t ---
g = arch_model(r * 100, mean='Constant', vol='GARCH', p=1, q=1, dist='t')
gfit = g.fit(disp='off', show_warning=False)
mu_g    = gfit.params['mu'] / 100
sigma_t = np.asarray(gfit.conditional_volatility) / 100
z_resid = (r - mu_g) / sigma_t
sigma_next = np.sqrt(gfit.forecast(horizon=1, reindex=False).variance.values[-1, 0]) / 100

# --- 1c MC t ---
np.random.seed(42)
sim_t = t_dist.rvs(df_t, loc=loc_t, scale=scale_t, size=200_000)

# --- VaR / EVaR / ES ---
def expectile(data, tau, weights=None):
    data = np.asarray(data)
    if weights is None:
        weights = np.ones_like(data) / len(data)
    else:
        weights = weights / weights.sum()
    def obj(e):
        up = np.sum(weights * np.maximum(data - e, 0.0))
        dn = np.sum(weights * np.maximum(e - data, 0.0))
        return tau * up - (1 - tau) * dn
    from scipy.optimize import brentq
    lo, hi = data.min() - 1, data.max() + 1
    return brentq(obj, lo, hi)

def es_emp(returns, alpha):
    q = np.percentile(returns, alpha * 100)
    return -returns[returns <= q].mean()

def es_t_analytic(alpha, df, mu, sigma):
    q = t_dist.ppf(alpha, df)
    pdf_q = t_dist.pdf(q, df)
    factor = (df + q**2) / (df - 1) * (pdf_q / alpha)
    return -(mu + sigma * (-factor))

def es_norm_analytic(alpha, mu, sigma):
    q = norm.ppf(alpha)
    return -(mu - sigma * norm.pdf(q) / alpha)

alphas = [0.05, 0.01]
rows_var = []
for a in alphas:
    # VaR
    var_param_t  = -t_dist.ppf(a, df_t, loc=loc_t, scale=scale_t)
    var_param_n  = -norm.ppf(a, loc=mu_n, scale=sig_n)
    var_hs       = -np.percentile(r, a*100)
    var_whs      = -wquant(r, w_brw, a)
    var_fhs      = -(mu_g + sigma_next * np.percentile(z_resid, a*100))
    var_mc       = -np.percentile(sim_t, a*100)
    # EVaR
    ev_pt        = -expectile(r, a)  # samples
    ev_hs        = -expectile(r, a)
    ev_whs       = -expectile(r, a, weights=w_brw)
    ev_fhs       = -(mu_g + sigma_next * expectile(z_resid, a))
    ev_mc        = -expectile(sim_t, a)
    # ES
    es_pt        = es_t_analytic(a, df_t, loc_t, scale_t)
    es_n         = es_norm_analytic(a, mu_n, sig_n)
    es_hs        = es_emp(r, a)
    z_tail       = z_resid[z_resid <= np.percentile(z_resid, a*100)]
    es_fhs       = -(mu_g + sigma_next * z_tail.mean())
    es_mc        = es_emp(sim_t, a)
    rows_var.append({
        'alpha':          f'{int((1-a)*100)}%',
        'Param Normal':   var_param_n,
        'Param t':        var_param_t,
        'Plain HS':      var_hs,
        'Weighted HS':      var_whs,
        'FHS GARCH':      var_fhs,
        'MC t':           var_mc,
        '_es_pt':         es_pt,
        '_es_n':          es_n,
        '_es_hs':         es_hs,
        '_es_fhs':        es_fhs,
        '_es_mc':         es_mc,
        '_ev_pt':         ev_pt,
        '_ev_whs':        ev_whs,
        '_ev_fhs':        ev_fhs,
        '_ev_mc':         ev_mc,
    })
df_var = pd.DataFrame(rows_var).set_index('alpha')

print('VaR — XTB.WA (daily log-returns, full sample 2018-2025):\n')
display(df_var[['Param Normal','Param t','Plain HS','Weighted HS','FHS GARCH','MC t']]
        .style.format('{:.4f}').background_gradient(cmap='Reds', axis=1))


In [ ]:
# --- Comparison chart VaR vs ES vs EVaR for 95% and 99% ---
methods   = ['Param Normal', 'Param t', 'Plain HS', 'Weighted HS', 'FHS GARCH', 'MC t']
keys_es   = ['_es_n', '_es_pt', '_es_hs', '_es_pt', '_es_fhs', '_es_mc']  # ES — Weighted HS ~ HS
keys_evar = ['_ev_pt', '_ev_pt', '_ev_pt', '_ev_whs', '_ev_fhs', '_ev_mc']

fig, axes = plt.subplots(1, 2, figsize=(17, 5.5), sharey=False)
for ax, a in zip(axes, alphas):
    var_v  = [df_var.loc[f'{int((1-a)*100)}%', m] for m in methods]
    es_v   = [df_var.loc[f'{int((1-a)*100)}%', k] for k in keys_es]
    evar_v = [df_var.loc[f'{int((1-a)*100)}%', k] for k in keys_evar]
    x = np.arange(len(methods))
    w = 0.27
    b1 = ax.bar(x - w, var_v,  width=w, color=XTB_STEEL,  label='VaR',  edgecolor='white')
    b2 = ax.bar(x,     es_v,   width=w, color=XTB_RED,    label='ES',   edgecolor='white')
    b3 = ax.bar(x + w, evar_v, width=w, color=XTB_ORANGE, label='EVaR', edgecolor='white')
    for bars in (b1, b2, b3):
        for b in bars:
            ax.text(b.get_x() + b.get_width()/2, b.get_height() + 0.002,
                    f'{b.get_height():.3f}', ha='center', va='bottom', fontsize=8.2)
    ax.set_xticks(x)
    ax.set_xticklabels(methods, rotation=18, ha='right')
    ax.set_title(f'Confidence level {int((1-a)*100)}%')
    ax.yaxis.set_major_formatter(PercentFormatter(xmax=1.0, decimals=1))
    ax.legend(loc='upper left')
fig.suptitle('VaR, ES and EVaR comparison — XTB.WA, full sample',
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()


### Conclusions — VaR/EVaR method comparison

1. **Param Normal underestimates VaR 99% by ~20%** relative to Param t and empirical HS.
   The gap widens at higher quantiles — Cornish-Fisher (Section 4.2 of prez4) showed
   that skewness and kurtosis correction gives VaR99 ≈ **25%** (sic!) instead of 6.7% from the normal —
   a numerically unstable extrapolation, but **the direction is clear**.

2. **FHS GARCH gives the lowest point estimates** (VaR99 ≈ 5.5%, ES99 ≈ 8.9%) **in a calm
   end-of-sample period** — because it forecasts VaR at the *current* volatility level, not the average
   over the last 8 years. This is precisely why FHS passes backtests (Section 5),
   while Plain HS does not.

3. **EVaR > VaR systematically** — the expectile incorporates the tail in a weighted average, not only
   at the boundary. For 99%: differences of 1–6 pp depending on method. EVaR is the only measure that is
   simultaneously *coherent and elicitable* — the FRTB direction.

4. **MC t = simulation equivalent of Param t** — differences are on the order of simulation noise (N=200k).
   MC makes sense only for multi-day scenarios where no closed-form formula exists.

5. **Bootstrap CI for VaR99 Param t** (from prez4): [0.073, 0.092] — CI width = **19 bp**
   (relatively 24% of the point estimate). Point VaR99 **is not enough** —
   reporting should include intervals.


---
## 5. VaR backtesting (rolling window) + Basel Traffic Light

From prez4 (rolling W=500 days, 1530 forecasts, 2020-2025):

| Method | Obs. frequency 99% | Kupiec p | Christ. p | Decision |
|---|---|---|---|---|
| **Param Normal**  | ~2.2% | ~0 | ~0 | REJECT |
| **Param t**       | ~1.8%   | 0.02 | 0.05 | marginal |
| **Plain HS**     | ~1.6%   | 0.07 | < 0.01 | lack of independence |
| **Weighted HS**     | ~1.3%   | 0.27 | 0.02 | lack of independence |
| **FHS GARCH**     | ~1.1%   | 0.65 | **0.36** | OK |
| **EWMA + Hill**   | ~1.0%   | 0.95 | **0.33** | **BEST** |

Selection of the *best method* was based on the product of CC (boot) p-values at 95% and 99%
— **EWMA + Hill** won (semi-parametric combination of dynamic volatility and tail index
estimator). FHS GARCH ranks just behind it.


In [ ]:
# --- Simulation of sample Basel tests for a random method (FHS-like) ---
# We reproduce a simplified backtest for 2-3 methods on the full XTB.WA sample
# (rolling W = 250 days, for speed); results are illustrative for visualization.

W_bt = 250
n_total = len(r)
dates_bt = log_ret.index[W_bt:]
actuals  = r[W_bt:]

# 3 methods for comparison: Normal, t-Student, FHS-GARCH (refit every 60 days)
fcst = {'Param Normal': [], 'Param t': [], 'FHS GARCH': []}
g_cache = None
for t_i in range(W_bt, n_total):
    win = r[t_i - W_bt:t_i]
    mu_w, sig_w = win.mean(), win.std()
    fcst['Param Normal'].append(-norm.ppf(0.01, loc=mu_w, scale=sig_w))
    try:
        dft_, lt_, st_ = t_dist.fit(win)
    except Exception:
        dft_, lt_, st_ = 5.0, mu_w, sig_w
    fcst['Param t'].append(-t_dist.ppf(0.01, dft_, loc=lt_, scale=st_))
    if (t_i - W_bt) % 60 == 0:
        try:
            gm = arch_model(win * 100, mean='Constant', vol='GARCH', p=1, q=1, dist='t')
            gf = gm.fit(disp='off', show_warning=False)
            g_mu = gf.params['mu'] / 100
            g_sig = np.asarray(gf.conditional_volatility) / 100
            g_z = (win - g_mu) / g_sig
            g_sigma_next = np.sqrt(gf.forecast(horizon=1, reindex=False).variance.values[-1, 0]) / 100
            g_cache = (g_mu, g_sigma_next, g_z)
        except Exception:
            g_cache = (mu_w, sig_w, (win - mu_w)/sig_w)
    g_mu, g_sn, g_z = g_cache
    fcst['FHS GARCH'].append(-(g_mu + g_sn * np.percentile(g_z, 1.0)))

for k in fcst: fcst[k] = np.array(fcst[k])

# Chart: return series + 3 VaR99 forecasts + breaches (3 panels)
fig, axes = plt.subplots(3, 1, figsize=(16, 11), sharex=True)
method_colors = {'Param Normal': XTB_STEEL,
                 'Param t':      XTB_ORANGE,
                 'FHS GARCH':    XTB_RED}
for ax, (m, v) in zip(axes, fcst.items()):
    viol = actuals < -v
    ax.plot(dates_bt, actuals, color=XTB_GRAPHITE, lw=0.55, alpha=0.8, label='log-return R_t')
    ax.plot(dates_bt, -v, color=method_colors[m], lw=1.4,
            label=f'-VaR 99% ({m})')
    ax.scatter(dates_bt[viol], actuals[viol], color=method_colors[m],
               edgecolor='white', linewidth=0.4, s=30, zorder=5,
               label=f'Breaches: {int(viol.sum())} ({viol.mean():.2%})')
    ax.axhline(0, color=XTB_GRAPHITE, lw=0.5)
    ax.set_title(f'{m} — rolling backtest VaR 99% (expected: ~{int(0.01*len(viol))})')
    ax.yaxis.set_major_formatter(PercentFormatter(xmax=1.0, decimals=0))
    ax.legend(loc='lower left', ncol=3, fontsize=9)
fig.suptitle(f'Backtest VaR 99% — XTB.WA, rolling window {W_bt} days',
             fontsize=14, fontweight='bold', y=1.00)
plt.tight_layout(); plt.show()


In [ ]:
# --- Basel Traffic Light test: number of breaches / year for 3 methods ---
def basel_zone(n_exc, n_obs):
    g = 4 * n_obs / 250
    y = 9 * n_obs / 250
    if n_exc <= g:   return 'GREEN', 3.00
    elif n_exc <= y: return 'YELLOW',   3.50
    else:            return 'RED', 4.00

basel_rows = []
ret_bt = pd.Series(actuals, index=dates_bt, name='R')
for name, v in fcst.items():
    vf = pd.Series(v, index=dates_bt)
    for year, grp in ret_bt.groupby(ret_bt.index.year):
        r_y = grp.values
        v_y = vf.loc[grp.index].values
        n_y = len(r_y)
        if n_y < 50: continue
        n_exc = int(np.sum(r_y < -v_y))
        zone, mult = basel_zone(n_exc, n_y)
        basel_rows.append({'Method': name, 'Year': int(year),
                           'N days': n_y, 'Violations': n_exc,
                           'Zone': zone, 'Multiplier': mult})
df_basel = pd.DataFrame(basel_rows)

# Heatmap chart: X-axis = years, Y-axis = methods, color = zone, label = breach count
zone_color = {'GREEN': XTB_GREEN, 'YELLOW': '#F59E0B', 'RED': XTB_RED}
years = sorted(df_basel['Year'].unique())
methods_b = ['Param Normal', 'Param t', 'FHS GARCH']

fig, ax = plt.subplots(figsize=(13, 4.5))
for i, m in enumerate(methods_b):
    for j, y in enumerate(years):
        sel = df_basel[(df_basel['Method'] == m) & (df_basel['Year'] == y)]
        if sel.empty: continue
        s = sel.iloc[0]
        ax.add_patch(plt.Rectangle((j-0.45, i-0.45), 0.9, 0.9,
                                   facecolor=zone_color[s['Zone']], alpha=0.85,
                                   edgecolor='white', linewidth=1.5))
        ax.text(j, i, f"{s['Violations']}\nx{s['Multiplier']:.2f}",
                ha='center', va='center', fontsize=10, fontweight='bold',
                color='white')
ax.set_xlim(-0.6, len(years)-0.4)
ax.set_ylim(-0.6, len(methods_b)-0.4)
ax.set_xticks(range(len(years))); ax.set_xticklabels(years)
ax.set_yticks(range(len(methods_b))); ax.set_yticklabels(methods_b)
ax.set_title('Basel Traffic Light — VaR99 breaches by year (label: N breaches / multiplier)')
ax.invert_yaxis()
ax.grid(False)
# Legenda
from matplotlib.patches import Patch
legend_elems = [Patch(facecolor=XTB_GREEN, label='GREEN: 0-4 breaches'),
                Patch(facecolor='#F59E0B', label='YELLOW: 5-9'),
                Patch(facecolor=XTB_RED, label='RED: >=10')]
ax.legend(handles=legend_elems, loc='upper right', bbox_to_anchor=(1.18, 1))
plt.tight_layout(); plt.show()

display(df_basel.pivot_table(index='Method', columns='Year',
                              values='Violations', aggfunc='sum').fillna('-'))


### Conclusions — backtesting and Basel

1. **Param Normal always has the most breaches** — in some years several times more
   than the expected 1%, which would result in a RED zone (capital multiplier 4.0 instead of 3.0).
   This is a real regulatory cost: for a 1M PLN position that is **~250k PLN of additional
   required capital** (Section 4.8 of prez4).

2. **FHS GARCH and EWMA+Hill consistently stay in the GREEN zone** in most years
   — confirming that **dynamic volatility methods are essential** for an asset with such strong
   clustering as XTB.WA.

3. **Plain HS fails the Christoffersen (independence) test** — breaches cluster
   in turbulent periods (COVID 2020, war 2022, 2023–2024 fluctuations linked to XTB earnings).
   This means the model **does not respond to risk when risk rises** — the biggest
   structural criticism of HS.

4. **The Berkowitz test (Section 2.2 of prez4)** additionally shows that FHS GARCH and EWMA+Hill have
   PIT closest to U(0,1) — i.e. **the entire predictive distribution is well calibrated**,
   not only the quantile itself.

5. **Recommendation for XTB:** use FHS GARCH or EWMA+Hill as the primary IMA model,
   with FHS-LSTM as an additional layer for even better 1% quantile calibration.


---
## 6. Extreme Value Theory (EVT) — GEV and GPD

Classical VaR approaches rely on the **full distribution** or its *quantile*. EVT focuses
exclusively on the **tail** — thus handling extreme losses better
(99.9% scenarios that occurred several times in 2018–2025).

- **Block Maxima (GEV):** split data into monthly blocks, take the maximum loss in each block;
  fit a GEV distribution with shape parameter **ξ**.
- **Peaks Over Threshold (POT, GPD):** all exceedances above a high threshold → Pareto distribution.

Shape parameter **ξ** classifies the tail:
- **ξ > 0** — Fréchet, heavy (power-law) tail
- **ξ = 0** — Gumbel, exponential tail
- **ξ < 0** — Weibull, bounded tail


In [ ]:
# --- 1) Block Maxima for XTB.WA losses + GEV ---
losses = -log_ret
monthly_max = losses.resample('ME').max().dropna()
c_gev, loc_gev, scale_gev = genextreme.fit(monthly_max)

# --- 2) Peaks Over Threshold + GPD ---
threshold = float(np.percentile(losses, 92.5))
exc = losses[losses > threshold] - threshold
c_gpd, _, scale_gpd = genpareto.fit(exc, floc=0)
n_exc, n_tot = len(exc), len(losses)

# --- VaR / ES z GEV i GPD ---
def var_gpd(alpha, u, xi, beta, n, nu):
    return u + (beta/xi) * ((n/nu*alpha)**(-xi) - 1)
def es_gpd(alpha, u, xi, beta, n, nu):
    v = var_gpd(alpha, u, xi, beta, n, nu)
    return (v + beta - xi*u) / (1 - xi)

evt_rows = []
for a in [0.05, 0.01, 0.001]:
    v_gev = float(genextreme.ppf(1 - a, c_gev,
                                  loc=loc_gev, scale=scale_gev))
    v_gpd = var_gpd(a, threshold, c_gpd, scale_gpd, n_tot, n_exc)
    e_gpd = es_gpd(a, threshold, c_gpd, scale_gpd, n_tot, n_exc)
    v_pt  = -t_dist.ppf(a, df_t, loc=loc_t, scale=scale_t)
    evt_rows.append({'alpha': f'{a*100:.1f}%',
                     'VaR GEV (mies.)': v_gev,
                     'VaR GPD (dz.)':   v_gpd,
                     'ES GPD (dz.)':    e_gpd,
                     'VaR Param-t (dz.)': v_pt})
df_evt = pd.DataFrame(evt_rows).set_index('alpha')
display(df_evt.style.format('{:.4f}'))

# --- Charts: GEV + GPD + mean excess ---
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('Extreme Value Theory for XTB.WA',
             fontsize=14, fontweight='bold', y=1.03)

# Panel 1: histogram block-maxima + GEV PDF
ax = axes[0]
ax.hist(monthly_max, bins=25, density=True, alpha=0.55,
        color=XTB_STEEL, edgecolor='white', label=f'Monthly maxima (n={len(monthly_max)})')
x_g = np.linspace(monthly_max.min()*0.9, monthly_max.max()*1.2, 300)
ax.plot(x_g, genextreme.pdf(x_g, c_gev, loc=loc_gev, scale=scale_gev),
        color=XTB_RED, lw=2.2, label=f'GEV (ξ={c_gev:.3f})')
ax.set_title('Block Maxima -> GEV')
ax.set_xlabel('Monthly loss')
ax.legend()

# Panel 2: histogram exceedances + GPD PDF
ax = axes[1]
ax.hist(exc, bins=30, density=True, alpha=0.55,
        color=XTB_ORANGE, edgecolor='white', label=f'Exceedances (n={n_exc})')
x_p = np.linspace(0, exc.max()*1.15, 300)
ax.plot(x_p, genpareto.pdf(x_p, c_gpd, scale=scale_gpd),
        color=XTB_RED, lw=2.2, label=f'GPD (ξ={c_gpd:.3f})')
ax.axvline(0, color=XTB_GRAPHITE, lw=0.5)
ax.set_title(f'Peaks Over Threshold (u = {threshold:.3f})')
ax.set_xlabel('Exceedance L - u')
ax.legend()

# Panel 3: mean excess plot
ax = axes[2]
thresholds_range = np.linspace(np.percentile(losses, 80),
                                np.percentile(losses, 99.5), 70)
me, valid = [], []
for u in thresholds_range:
    ee = losses[losses > u] - u
    if len(ee) > 5:
        me.append(ee.mean()); valid.append(u)
me = np.array(me); valid = np.array(valid)
me_th = (scale_gpd + c_gpd * (valid - threshold)) / (1 - c_gpd) if c_gpd < 1 else None
ax.scatter(valid, me, s=22, color=XTB_NAVY, alpha=0.7, label='empirical e(u)')
if me_th is not None:
    ax.plot(valid, me_th, color=XTB_RED, lw=1.8, label='theoretical (GPD)')
ax.axvline(threshold, color=XTB_GRAPHITE, ls='--', lw=1.2,
           label=f'Selected threshold = {threshold:.3f}')
ax.set_title('Mean Excess Plot (should be linear in u)')
ax.set_xlabel('Threshold u')
ax.set_ylabel('e(u) = E[L-u | L>u]')
ax.legend()

plt.tight_layout(); plt.show()


### Conclusions — EVT

1. **GPD shape parameter ξ ~ 0.34–0.6** → XTB.WA has **Fréchet-type** distribution tails:
   probabilities of extreme losses decay slowly (power law), not exponentially.
   The right diagnosis for financial assets with "black swan" events.

2. **Mean Excess Plot rises linearly** in the exceedance region > 92.5% quantile — graphical
   confirmation that GPD is the correct tail model in this range.
   Empirical/theoretical correlation coefficient ~0.98 (from prez3).

3. **VaR and ES from GPD give significantly higher estimates in extreme tails** (99.9%):
   GPD VaR99.9 ≈ 18.4%, ES99.9 ≈ 28.8%, while Param-t VaR99.9 ≈ 16%.
   For stress-test scenarios (ECB, FRTB IMA) **GPD is the standard**.

4. **GEV (block maxima) confirms** Fréchet character, but the number of blocks (96 months)
   is small, resulting in wide CIs for parameters.
   **POT/GPD uses more tail observations** (153 vs 96) → more stable estimation.

5. **Practice for XTB:**
   - daily VaR and IMA capital requirements → FHS GARCH or LSTM-FHS (well calibrated);
   - capital reserve for extreme scenarios (Pillar II / EBA stress test) → GPD;
   - combining both gives full risk coverage — from daily operations to tail risk.


---
## 7. Hybrid LSTM + FHS model (`lstm_fhs_xtb.py`)

In the final project stage, a machine learning model was added — **LSTM** forecasting
volatility, combined with **Filtered Historical Simulation** for tail estimation.

**Idea:**
$$
\widehat{\mathrm{VaR}}_{\alpha,\,t+1} = -\hat{\sigma}_{t+1}^{\text{LSTM}} \cdot Q_\alpha(z),
$$
where:
- **sigma_LSTM** — forecast from LSTM(32) retrained every 90 days (rolling refit),
- **Q_alpha(z)** — empirical quantile of standardized residuals z_t = r_t / sigma_LSTM_t.

**Motivation:**
1. XTB.WA returns have negative skew → empirical quantile natively handles tail asymmetry;
2. no parametric assumption on residual distribution → flexible capture of fat tails;
3. LSTM uses 6 features (log-returns, volume, EWMA, sigma_close, etc.), not only
   r² autocorrelation as in GARCH.

**Grid search** (run in the cell below via `lstm_fhs_xtb.py`, ~5–10 min):
- WINDOW (LSTM sequence length): 30 / 50 / 80
- LAMBDA_EWMA (EWMA feature): 0.94 / 0.97
- SIGMA_FLOOR (EWMA multiplier as lower bound on sigma): 0 - 0.9
- RESID_WIN (residual window for quantile): 250-1500 / all


In [ ]:
# --- LSTM-FHS: full replication of lstm_fhs_xtb.py (grid + best configuration) ---
# Note: LSTM training ~5-10 min on MPS/CPU (6 WINDOW x LAMBDA pairs).
import importlib
import lstm_fhs_xtb as lf

importlib.reload(lf)
lstm_out = lf.run_grid_search(verbose_pipeline=True, verbose_grid=True)

n_te = len(lstm_out['r_actual'])
print(f'\nTest set: N = {n_te} days (80% train / 20% test)')
print(f'Expected breaches: 95% -> {int(round(0.05 * n_te))}, 99% -> {int(round(0.01 * n_te))}')

print('\n--- Configurations passing Kupiec + Christoffersen (95% and 99%) ---')
display(lstm_out['df_pass'])

best = lstm_out['best_row']
t95 = lstm_out['best_res']['tests'][0.05]
t99 = lstm_out['best_res']['tests'][0.01]
print('\nBest configuration (same selection logic as in lstm_fhs_xtb.py):')
print(f'  WINDOW={lstm_out["best_win"]}, LAMBDA_EWMA={lstm_out["best_lam"]}, '
      f'SIGMA_FLOOR={lstm_out["best_sf"]}, '
      f'RW95={lf._fmt_rw(lstm_out["best_rw95"])}, RW99={lf._fmt_rw(lstm_out["best_rw99"])}')
print(f'  95%: n={t95["n_viol"]}, freq={t95["freq"]:.2%}, Kupiec p={t95["p_kup"]:.3f}, '
      f'Christoffersen p={t95["p_chr"]:.3f}')
print(f'  99%: n={t99["n_viol"]}, freq={t99["freq"]:.2%}, Kupiec p={t99["p_kup"]:.3f}, '
      f'Christoffersen p={t99["p_chr"]:.3f}')


In [ ]:
# --- LSTM-FHS charts: data from lstm_out (best configuration after grid search) ---
o = lstm_out
dates_te = o['dates_te']
r_actual = o['r_actual']
sigma_pred = o['sigma_pred']
sigma_pred_raw = o['sigma_pred_raw']
ewma_te = o['ewma_te']
vars_pred = o['vars_pred']
df_best = o['df_best']
n_train = o['n_train']

fig, axes = plt.subplots(2, 1, figsize=(16, 8), sharex=True)
fig.suptitle(
    f'LSTM-FHS — XTB.WA (WIN={o["best_win"]}, LAM={o["best_lam"]}, SF={o["best_sf"]}, '
    f'RW95={lf._fmt_rw(o["best_rw95"])}, RW99={lf._fmt_rw(o["best_rw99"])})',
    fontsize=14, fontweight='bold', y=1.00,
)

# Panel 1: volatility forecast (rolling refit)
ax = axes[0]
ax.plot(dates_te, sigma_pred, color=XTB_RED, lw=1.1,
        label=f'sigma LSTM (floor={o["best_sf"]})')
ax.plot(dates_te, sigma_pred_raw, color=XTB_STEEL, lw=0.7, alpha=0.45,
        label='sigma LSTM (raw)')
ax.plot(dates_te, ewma_te, color=XTB_NAVY, lw=0.9, alpha=0.75,
        label=f'EWMA sigma (lambda={o["best_lam"]})')
for k in range(n_train + lf.REFIT_STEP, len(df_best), lf.REFIT_STEP):
    ax.axvline(df_best.index[k], color=XTB_GRAPHITE, ls=':', lw=0.5, alpha=0.35)
ax.set_title('Volatility forecast $\\sigma_{t+1}$ (rolling refit every 90 days)')
ax.set_ylabel('$\\sigma_{t+1}$')
ax.yaxis.set_major_formatter(PercentFormatter(xmax=1.0, decimals=0))
ax.legend(loc='upper right', ncol=3, fontsize=9)

# Panel 2: VaR 95% and 99% + breaches (out-of-sample)
ax = axes[1]
ax.plot(dates_te, r_actual, color=XTB_GRAPHITE, lw=0.55, alpha=0.8, label='$R_t$')
for a, c in zip(lf.ALPHAS, [XTB_ORANGE, XTB_RED]):
    var_a = vars_pred[a]
    viol = r_actual < -var_a
    ax.plot(dates_te, -var_a, color=c, lw=1.3,
            label=f'-VaR {int((1 - a) * 100)}% (FHS)')
    ax.scatter(dates_te[viol], r_actual[viol], color=c,
               edgecolor='white', linewidth=0.4, s=28, zorder=5,
               label=f'Breaches {int((1 - a) * 100)}%: {int(viol.sum())} ({viol.mean():.2%})')
ax.axhline(0, color=XTB_GRAPHITE, lw=0.5)
ax.set_title('Dynamic VaR vs log-returns (test set)')
ax.yaxis.set_major_formatter(PercentFormatter(xmax=1.0, decimals=0))
ax.legend(loc='lower left', ncol=3, fontsize=9)

plt.tight_layout()
plt.show()


### Conclusions — LSTM + FHS

1. **The best configuration passed all tests** (Kupiec + Christoffersen) at both
   levels (95% and 99%) with p-values close to 1.0. This is a rarely seen result
   for XTB.WA — even FHS-GARCH has Christoffersen p ≈ 0.36, while LSTM-FHS reaches 0.78–1.00.

2. **Short LSTM sequence (WINDOW=30) beat** longer ones (50, 80). Interpretation:
   for XTB.WA *the signal from the last 6 weeks matters most*; longer history adds nothing
   and introduces noise.

3. **RW95=250 / RW99=all** — shorter window for 5% adapts better to the current regime,
   long window for 1% is necessary for stability of rare quantiles. This reflects the
   **bias-variance tradeoff** in EVT hyperparameter choice.

4. **SIGMA_FLOOR=0.0 is optimal** — LSTM alone produces a sensible forecast; additional
   EWMA floor lowers accuracy during regime changes.

5. **Methodological lessons (LSTM + FHS vs GARCH):**
   - LSTM captures **nonlinearities and feature interactions** (e.g. volume signals rising
     volatility before sigma — an effect absent in GARCH);
   - FHS guards against parametric errors — empirical quantile of residuals is **robust to
     tail asymmetry**;
   - the combination yields *dynamic sigma + nonparametric tail* = best of both worlds.

6. **Computational cost:** LSTM-FHS requires ~5–10 min training on MPS (Apple Silicon).
   Acceptable for student projects; production would require GPU.


---
## 8. Markowitz portfolio optimization (prez5)

Task: allocate **1M PLN** across WSE equities (13 companies, including WIG20 blue chips).

We determine:
- **MVP (Minimum Variance Portfolio)** — minimum-risk portfolio;
- **market portfolio (max Sharpe)** — tangency portfolio on the CML;
- **efficient frontier** (sweep over target return);
- **Monte Carlo cloud** — 10,000 random portfolios from a Dirichlet distribution;
- comparison with the **single-factor model** (Sigma_SIM vs Sigma_sample).


In [ ]:
# --- Replication of prez5 with 13 WIG20 + peer stocks ---
TICKERS_5 = ['PKO.WA', 'PKN.WA', 'KGH.WA', 'CDR.WA', 'DNP.WA',
             'VOT.WA', 'KRU.WA', 'PZU.WA', 'ENT.WA', 'DVL.WA',
             'DOM.WA', 'ALE.WA', 'ACP.WA']
NAMES = {'PKO.WA':'PKO BP','PKN.WA':'PKN Orlen','KGH.WA':'KGHM',
         'CDR.WA':'CD Projekt','DNP.WA':'Dino','VOT.WA':'VOTUM',
         'KRU.WA':'Kruk','PZU.WA':'PZU','ENT.WA':'Enter Air',
         'DVL.WA':'Develia','DOM.WA':'Dom Dev.','ALE.WA':'Allegro',
         'ACP.WA':'Asseco'}
START_P, END_P = '2021-01-02', '2026-05-13'
RF_ANN = 0.05
TD = 252
CAPITAL = 1_000_000

raw = yf.download(TICKERS_5, start=START_P, end=END_P, progress=False, auto_adjust=False)
if isinstance(raw.columns, pd.MultiIndex):
    field = 'Adj Close' if 'Adj Close' in raw.columns.get_level_values(0) else 'Close'
    stocks = raw[field][TICKERS_5].dropna(how='all')
else:
    stocks = raw[['Close']].copy(); stocks.columns = TICKERS_5
stocks.columns = [NAMES[c] for c in stocks.columns]

stocks = stocks.dropna()
log_ret_p = np.log(stocks / stocks.shift(1)).dropna()
mu5 = log_ret_p.mean().values * TD
Sigma5 = log_ret_p.cov().values * TD
names5 = list(log_ret_p.columns)
n_assets = len(names5)

# Optimization: MVP + max Sharpe + frontier
def port_var(w, S): return w @ S @ w
def port_ret(w, m): return w @ m
def neg_sharpe(w, m, S, rf): return -(w @ m - rf) / np.sqrt(w @ S @ w)

bounds = tuple((0.0, 1.0) for _ in range(n_assets))
cons_sum = {'type':'eq', 'fun': lambda w: w.sum() - 1.0}

res_mvp = minimize(port_var, x0=np.ones(n_assets)/n_assets, args=(Sigma5,),
                    method='SLSQP', bounds=bounds, constraints=[cons_sum])
w_mvp = res_mvp.x
mu_mvp, sig_mvp = w_mvp @ mu5, np.sqrt(w_mvp @ Sigma5 @ w_mvp)

res_mkt = minimize(neg_sharpe, x0=np.ones(n_assets)/n_assets,
                    args=(mu5, Sigma5, RF_ANN), method='SLSQP',
                    bounds=bounds, constraints=[cons_sum])
w_mkt = res_mkt.x
mu_mkt, sig_mkt = w_mkt @ mu5, np.sqrt(w_mkt @ Sigma5 @ w_mkt)
S_mkt = (mu_mkt - RF_ANN) / sig_mkt

# Efficient frontier
mu_targets = np.linspace(mu_mvp, mu5.max(), 50)
eff_sig, eff_mu = [], []
for mt in mu_targets:
    cons = [cons_sum, {'type':'eq', 'fun': lambda w, mt=mt: w @ mu5 - mt}]
    res = minimize(port_var, x0=np.ones(n_assets)/n_assets, args=(Sigma5,),
                    method='SLSQP', bounds=bounds, constraints=cons)
    if res.success:
        eff_sig.append(np.sqrt(port_var(res.x, Sigma5)))
        eff_mu.append(mt)
eff_sig, eff_mu = np.array(eff_sig), np.array(eff_mu)

# MC: 10,000 random portfolios
np.random.seed(42)
W_mc = np.random.dirichlet(np.ones(n_assets), size=10000)
mc_mu  = W_mc @ mu5
mc_sig = np.sqrt(np.einsum('ij,jk,ik->i', W_mc, Sigma5, W_mc))
mc_S   = (mc_mu - RF_ANN) / mc_sig

print(f'MVP portfolio:       E(R)={mu_mvp:.4f}, sigma={sig_mvp:.4f}, Sharpe={(mu_mvp-RF_ANN)/sig_mvp:.4f}')
print(f'Market portfolio:   E(R)={mu_mkt:.4f}, sigma={sig_mkt:.4f}, Sharpe={S_mkt:.4f}')


In [ ]:
# --- Chart: efficient frontier + MC cloud + two optimal portfolios ---
fig, ax = plt.subplots(figsize=(13, 8))

sc = ax.scatter(mc_sig, mc_mu, c=mc_S, cmap='viridis', s=10, alpha=0.45,
                edgecolor='none')
cbar = plt.colorbar(sc, ax=ax, fraction=0.046, pad=0.02)
cbar.set_label('Sharpe ratio', fontsize=11)

# Efficient frontier
ax.plot(eff_sig, eff_mu, color=XTB_RED, lw=3, label='Efficient frontier',
        zorder=4, solid_capstyle='round')

# CML
x_cml = np.linspace(0, mc_sig.max()*1.05, 100)
ax.plot(x_cml, RF_ANN + (mu_mkt - RF_ANN)/sig_mkt * x_cml,
        ls='--', color=XTB_RED, alpha=0.6, lw=1.5, label=f'CML (Sharpe={S_mkt:.2f})')

# Individual stocks
for i, name in enumerate(names5):
    sig_i = np.sqrt(Sigma5[i,i])
    ax.scatter(sig_i, mu5[i], s=80, marker='s', edgecolor=XTB_INK,
               linewidth=0.8,
               facecolor=XTB_PALETTE[i % len(XTB_PALETTE)], zorder=5)
    ax.annotate(name, (sig_i, mu5[i]),
                xytext=(7, 4), textcoords='offset points',
                fontsize=8.5, fontweight='bold', color=XTB_INK)

# Optimal portfolios
ax.scatter(sig_mvp, mu_mvp, s=350, marker='*', color=XTB_NAVY,
           edgecolor='white', linewidth=2,
           label=f'MVP (sigma={sig_mvp:.3f})', zorder=7)
ax.scatter(sig_mkt, mu_mkt, s=350, marker='*', color=XTB_RED,
           edgecolor='white', linewidth=2,
           label=f'Market portfolio (Sharpe={S_mkt:.2f})', zorder=7)

ax.axhline(RF_ANN, color=XTB_GRAPHITE, ls=':', alpha=0.6, lw=1,
           label=f'r_f = {RF_ANN:.1%}')
ax.set_xlim(left=0)
ax.set_xlabel('Risk σ_p (annual)')
ax.set_ylabel('Expected return E(R_p)')
ax.set_title('Efficient frontier for 13 WSE stocks — Markowitz portfolios',
             fontweight='bold', fontsize=13)
ax.xaxis.set_major_formatter(PercentFormatter(xmax=1.0, decimals=0))
ax.yaxis.set_major_formatter(PercentFormatter(xmax=1.0, decimals=0))
ax.legend(loc='lower right', fontsize=10)
plt.tight_layout(); plt.show()


In [ ]:
# --- 1M PLN allocation: MVP portfolio vs market portfolio ---
alloc = pd.DataFrame({
    'w_MVP':            np.round(w_mvp, 4),
    'PLN_MVP':          np.round(w_mvp * CAPITAL, 0),
    'w_Market':         np.round(w_mkt, 4),
    'PLN_Market':       np.round(w_mkt * CAPITAL, 0),
}, index=names5)
alloc = alloc.loc[(alloc[['w_MVP','w_Market']].sum(axis=1) > 1e-4)]
display(alloc.sort_values('w_Market', ascending=False)
             .style.format({'w_MVP':'{:.2%}', 'w_Market':'{:.2%}',
                            'PLN_MVP':'{:,.0f}', 'PLN_Market':'{:,.0f}'}))

# Bar chart: MVP vs Market allocation
fig, ax = plt.subplots(figsize=(14, 5))
sorted_idx = alloc['w_Market'].sort_values(ascending=False).index
x_pos = np.arange(len(sorted_idx))
w_bar = 0.4
ax.bar(x_pos - w_bar/2, alloc.loc[sorted_idx, 'w_MVP'],     width=w_bar,
       color=XTB_NAVY,  label='MVP', edgecolor='white')
ax.bar(x_pos + w_bar/2, alloc.loc[sorted_idx, 'w_Market'],  width=w_bar,
       color=XTB_RED,   label='Market portfolio', edgecolor='white')
ax.set_xticks(x_pos)
ax.set_xticklabels(sorted_idx, rotation=25, ha='right')
ax.set_ylabel('Portfolio weight')
ax.set_title('1M PLN capital allocation: MVP vs market portfolio (13 WSE stocks)')
ax.yaxis.set_major_formatter(PercentFormatter(xmax=1.0, decimals=0))
ax.legend()
plt.tight_layout(); plt.show()


### Conclusions — Markowitz

1. **Market portfolio has Sharpe ≈ 1.26** vs 0.79 for MVP — a clear premium for taking more
   exposure. With 5% annual risk-free and 31% expected return, **annual excess return**
   is 26 pp at 21% volatility.

2. **Top positions in the market portfolio:** Develia (30%), Dom Development (21%), Asseco (17%),
   VOTUM (19%), Kruk (10%), PKN Orlen (11%). This is a portfolio with **strong real-estate exposure**
   (Develia + Dom Dev = 51%) and defensive IT/PZU. A surprise is **zero allocation to
   PKO BP, KGHM, and CD Projekt** — the best-known blue chips.

3. **Diversification really works** — the efficient frontier for 13 stocks lies **clearly above**
   that for 2 (from prez5 Section 3): at 21% volatility we get 31% return instead of ~25% from a
   2-stock portfolio.

4. **Sigma_SIM matrix has Frobenius norm difference = 0.097**, but **real risk of portfolios
   based on SIM is 5–10% higher** than those based on sample Sigma.
   This is a trade-off: SIM regularizes weights (less extreme allocations) at the cost of higher
   realized sigma.

5. **Recommendation for XTB** (context of prez5 point 5):
   - for *risk-adjusted return* → market portfolio with sample Sigma;
   - with a *hard risk budget* (VaR99 ≤ 20%) → frontier portfolio with risk-free asset added
     (5% government bonds);
   - SIM = sanity check for systemic vs idiosyncratic risk allocation.


---
## 9. Synthesis of conclusions and recommendations for XTB S.A.

After going through all four materials, a coherent picture emerges of XTB.WA risk and how to
measure it.

### 9.1 What we know about XTB.WA

| Attribute | Value | Implication |
|---|---|---|
| Trading volume | high (WIG20) | 1M position can be liquidated in 1 day |
| Annual sigma   | ~47% | Classification: **high risk** (vs WIG20 ~22%) |
| Skewness       | -0.44 | Larger negative extremes than positive |
| Excess kurtosis   | 25.4 | Drastically fat tails (vs 0 for Normal) |
| ν (t-Student) | 2.56 | Even t-distribution has "barely finite" variance |
| GPD ξ        | 0.34–0.60 | Fréchet — power tail, "heavy" |
| AR(1) coef     | -0.12 | Weakly visible autoregression |

### 9.2 Method hierarchy (weakest to strongest)

```
Param Normal  <  Plain HS  <  Param t-Student  <  Weighted HS  <  FHS GARCH  <  EWMA + Hill  <  LSTM + FHS
  thin tail   |  no vol.    |  symmetric         | adaptive    | dynamic     |  semi-EVT     |  feature
              |  dynamics   |  tail              | regime      | sigma+tail  |  on tail      |  learning
```

Best methods: **LSTM-FHS** (ideal backtest), **EWMA+Hill** (simplest among
top-tier), **FHS GARCH** (industry standard).

### 9.3 Recommended risk stack for XTB

| Goal | Method | Rationale |
|---|---|---|
| **Daily reporting** | FHS GARCH | Standard, proven, easy to audit |
| **Capital requirement (IMA)** | LSTM-FHS or EWMA+Hill | Highest CC p-values in Christoffersen |
| **Stress test (Pillar II)** | GPD / POT | Best for 99.9% scenarios |
| **Strategic asset allocation** | Markowitz + SIM check | Low sensitivity to overfitting |
| **ICAAP narrative** | Cornish-Fisher + Bootstrap CI | Communicating uncertainty |

### 9.4 Methodological lessons

1. **A single risk measure is never enough.** Combine:
   - VaR (quantile) — easy to communicate,
   - ES (conditional tail mean) — coherent properties, FRTB,
   - EVaR (expectile) — the only coherent and elicitable measure.

2. **Backtesting is absolutely necessary.** Without Kupiec and Christoffersen we cannot
   claim the model is properly calibrated. The Berkowitz test additionally
   verifies the *entire* predictive distribution, not only the quantile.

3. **Method choice has real financial consequences.** The difference in required
   IMA capital for a 1M PLN position between Param Normal and FHS GARCH is about **250k PLN**.
   That is a real gain for XTB from better calibration.

4. **Machine learning (LSTM) does not replace, but complements GARCH.** For individual
   assets with strong dynamics (like XTB.WA) LSTM-FHS is significantly better calibrated,
   but requires longer history and mandatory audit obligations.

5. **Portfolio diversification** (5+ stocks from different sectors) delivers benefits comparable
   to optimization with advanced risk measures. **The simplest way to reduce risk
   without sacrificing return**.

---

## 9.5 Summary — key recommendations

> 1. **Use FHS GARCH or LSTM-FHS** as primary VaR/ES methods for XTB.WA.
> 2. **Never report VaR99 without a confidence interval** — bootstrap CI is standard.
> 3. **Use GPD** for extreme scenarios (99.9%, stress test).
> 4. **Diversify by sector** — portfolio of 5–10 WSE stocks from different sectors.
> 5. **Backtest quarterly** — Kupiec, Christoffersen, Berkowitz, Basel Traffic Light.
> 6. **Model EVaR and ES**, not only VaR — FRTB is the regulatory direction.

### Bibliography (used in projects)

- Basel Committee on Banking Supervision (1996): *Supervisory framework for the use of "backtesting"*.
- Berkowitz, J. (2001): *Testing density forecasts, with applications to risk management*.
- Bellini & Bignozzi (2015): *On elicitable risk measures*.
- Boudoukh, Richardson & Whitelaw (1998): *The best of both worlds*.
- Christoffersen, P.F. (1998): *Evaluating Interval Forecasts*.
- Diebold & Mariano (1995): *Comparing Predictive Accuracy*.
- Hill, B.M. (1975): *A simple general approach to inference about the tail of a distribution*.
- Kupiec, P.H. (1995): *Techniques for verifying the accuracy of risk measurement models*.
- Pickands (1975), Balkema & de Haan (1974): *POT / GPD theorem*.

---

*Notebook generated as a summary of projects 3–4 for the Enterprise Risk Management course, summer semester 2025/26.*
